# From Linear Convolution to Overlap-Add Method!

This notebook explains the math behind the overlap-add method in convolution. The content is based on the book "Discrete-Time Signal Processing" by Alan V. Oppenheim and Ronald W. Schafer. The book is [publicly available in MIT OpenCourseWare](https://ocw.mit.edu/courses/res-6-dtsp-discrete-time-signal-processing/resources/mitres_6-dtsp_s26_thirdedition_pdf/)

Most textbooks use $x$ and $h$ notation in the convolution. However, in this notebook, we would like to use $Q$ and $T$ as those are the name of arrays in the realm of STUMPY. We would also like to introduce $Q'$, which is simply the reverse of $Q$.

# Linear Convolution of Two Finite-Length Sequences

Consider two finte-length sequences $Q'$ (of length $m$) and $T$ (of length $n$). Their linear convolution, $C$, can be computed as follows:

$$ C[idx] = \sum_{i=-\infty}^{\infty}{T[i] \times Q'[idx-i]} $$

where $T[.]$ and $Q'[.]$ are zeros for any index that is outside of their range. So, $T[i]$ is zero if $i$ is outside of the range $0 \le i \le n-1$. Hence, the equation above is equivalent to:

$$ C[idx] = \sum_{i=0}^{n-1}{T[i] \times Q'[idx-i]} $$

Note that $Q'[idx-i]$ is not zero when $0 \le idx-i \le m-1$, or equivalently $i \le idx \le i+m-1$. 

Therefore:

* $idx \ge i, \quad i \ge 0 \implies idx \ge 0$
* $idx \le i+m-1, \quad i \le n-1 \implies idx \le n+m-2$

This shows that the linear convolution $C$ has values at indices $\set{0, 1, ..., n + m - 2}$, and it is 0 otherwise. Therefore, the length of output in linear convolution is $n+m-1$. 

Furthermore, let's check out the values for different indices:
* $idx=0 \implies C[0]=T[0]Q'[0]$
* $idx=1 \implies C[0]=T[0]Q'[1] + T[1]Q'[0]$
* ...
* $idx=m-1 \implies C[m-1]=T[0]Q'[m-1] + T[1]Q'[m-2] + ... + T[m-1]Q'[0] = T_{0}.Q$
* $idx=m \implies C[m]=T[1]Q'[m-1] + T[2]Q'[m-2] + ... + T[m]Q'[0] = T_{1}.Q$
* ...
* $idx=n-1 \implies C[n-1]=T[n-m]Q'[(n-1)-(n-m)] + T[n-m+1]Q'[(n-1)-(n-m+1)] + ... + T[n-1]Q'[0] = T_{n-m+1}.Q$
* ...
* $idx=n+m-2 \implies C[n+m-2]=T[n-1]Q'[(n+m-2)-(n-1)] = T[n-1]Q'[m-1]$

As observed, when $m-1 \le idx \le n-1$, $C[idx]$ becomes the dot product between a subsequnce of $T$ and $Q$, which is the reverse of $Q'$. Therefore, the `range(m-1,n)` gives the sliding dot product between $Q$ and $T$.

**How can we leverage this to speed up the computation of sliding dot product?**

# Option I: Convert to Circular Convolution and use FFT-IFFT

Circular convolution is defined between two sequences that are both periodic and their period are the same, say $N$. Their circular convolution is also a periodic sequence, with period $N$, and it can be computed as follows:

$$C_{N}[idx] = \sum_{i=0}^{N-1} \tilde{Q'}[i] \times \tilde{T}[(idx-i)_{N}], \quad 0 \le idx \le N-1$$

where, $\tilde{Q'}$ and $\tilde{T}$ are both periodic sequence with period $N$. $C_{N}$ represents one period of N-Circular convolution. This can also be computed via FFT, i.e. $$C_{N} = IFFT( FFT(\tilde{Q'}_{N}) \times FFT(\tilde{T}_{N}) ) $$

If there is a way to compute the linear convolution via circular convolution, then we can take advantage of FFT by using eq (4). The good news is that there is a way! The linear convolution between $Q'$ and $T$ can be obtained by performing N-circular convolution between $\tilde{Q'}$ and $\tilde{T}$, where:

* $N \ge n + m - 2$
* $\tilde{Q'}_{N}$ is $Q'$ but zero-padded with $N-m$ zeros
* $\tilde{T}_{N}$ is $T$ but zero-padded with $N-n$ zeros

In [2]:
# WIP

# Option II: Overlap-add method
This is a divide and conquer algorithm. It applies divide on "linear convolution" and conquer each via circular convolution (See: Option I)

## Linearity in Linear Convolution

Suppose the array $T$ can be written as $T_{1} + T_{2}$. In other words: $T[i] = T_{1}[i] + T_{2}[i]$, then:

$$ C[idx] = \sum_{i=-\infty}^{\infty}{T[i] \times Q'[idx-i]} $$

$$ C[idx] = \sum_{i=-\infty}^{\infty}{(T_{1}[i] \times Q'[idx-i] + T_{2}[i] \times Q'[idx-i])} $$

$$ C[idx] = \sum_{i=-\infty}^{\infty}{T_{1}[i] \times Q'[idx-i]} + \sum_{i=-\infty}^{\infty}{T_{2}[i] \times Q'[idx-i]}$$

$$ C[idx] = C_{1}[idx] + C_{2}[idx] $$

This shows that a linear convolution has the property of "linearity". Now, we use this property to show that we can compute the linear convolution between $Q'$ and long $T$ by breaking $T$ into smaller parts.